# Meta-learner (S/T/X-learner) 実データ実験 — クーポンキャンペーン

卒論8章の再現。データは `couponsretailer.csv`(処置=coupons_Any、結果=dailyspending)。
`metalearner_real/` パッケージの関数を呼ぶだけの薄いノートブック。

計算量の注意: 卒論と同じ設定(n_iter=100)はかなり時間がかかる。
まず動作を確認したいときは、下のセルの `N_ITER` を小さくして試すとよい。

In [ ]:
from metalearner_real.data_real import load_and_split, TREATMENT_COL, OUTCOME_COL
from metalearner_real.learners_real import (
    search_xgb, search_rf,
    fit_s_learner, predict_s_learner,
    fit_t_learner, predict_t_learner,
    fit_x_learner, predict_x_learner,
)
from metalearner_real.analysis_real import plot_cate_boxplot, plot_cate_violin, print_ate_summary
from metalearner_real.shap_utils_real import compute_ite_shap, plot_ite_shap

In [ ]:
# ここを書き換えれば計算量を調整できる(卒論はn_iter=100)
N_ITER = 100

train_df, test_df, cv5 = load_and_split("couponsretailer.csv")
print("train:", train_df.shape, " test:", test_df.shape)

## S-learner

In [ ]:
s_model_xgb, s_model_rf = fit_s_learner(train_df, cv=cv5, n_iter=N_ITER)
s_tau_xgb, s_mu0_xgb, s_mu1_xgb = predict_s_learner(s_model_xgb, test_df)
s_tau_rf, s_mu0_rf, s_mu1_rf = predict_s_learner(s_model_rf, test_df)

## T-learner

In [ ]:
t_model_0_xgb, t_model_1_xgb, t_model_0_rf, t_model_1_rf = fit_t_learner(train_df, n_iter=N_ITER)
t_tau_xgb, t_mu0_xgb, t_mu1_xgb = predict_t_learner(t_model_0_xgb, t_model_1_xgb, test_df)
t_tau_rf, t_mu0_rf, t_mu1_rf = predict_t_learner(t_model_0_rf, t_model_1_rf, test_df)

## X-learner(T-learnerのモデルを使って学習する)

In [ ]:
x_tau0_xgb, x_tau1_xgb, x_prop_xgb = fit_x_learner(train_df, t_model_0_xgb, t_model_1_xgb, search_fn=search_xgb, n_iter=N_ITER)
x_tau_xgb, _, _, _ = predict_x_learner(x_tau0_xgb, x_tau1_xgb, x_prop_xgb, test_df)

x_tau0_rf, x_tau1_rf, x_prop_rf = fit_x_learner(train_df, t_model_0_rf, t_model_1_rf, search_fn=search_rf, n_iter=N_ITER)
x_tau_rf, _, _, _ = predict_x_learner(x_tau0_rf, x_tau1_rf, x_prop_rf, test_df)

## 結果の可視化(箱ひげ図・バイオリンプロット・ブートストラップATE)

In [ ]:
tau_by_learner = {
    "S-learner (XGB)": s_tau_xgb,
    "S-learner (RF)": s_tau_rf,
    "T-learner (XGB)": t_tau_xgb,
    "T-learner (RF)": t_tau_rf,
    "X-learner (XGB)": x_tau_xgb,
    "X-learner (RF)": x_tau_rf,
}

plot_cate_boxplot(tau_by_learner)
plot_cate_violin(tau_by_learner)
print_ate_summary(tau_by_learner)

## SHAP値(ITEへの寄与度、X-learner XGBoostを例に)

In [ ]:
shap_values_ite, X_test = compute_ite_shap(
    x_tau0_xgb, x_tau1_xgb, x_prop_xgb, test_df,
    treatment_col=TREATMENT_COL, outcome_col=OUTCOME_COL,
)
plot_ite_shap(shap_values_ite, X_test)